# Import libraries

In [2]:
!pip install -q datasets sentence-transformers pymilvus pymilvus[milvus_lite] pyserini
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

In [3]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# Connect to Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Utility functions

In [5]:
from datasets import load_dataset
import json
import os

base_path = "/content/drive/MyDrive/CapstoneProject/Retriever"

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def load_qrels(path):
    qrels = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            qid, _, docid, rel = line.strip().split()
            qrels.setdefault(qid, {})[docid] = int(rel)
    return qrels

def normalize_scores(scores_dict):
    if not scores_dict: return {}
    vals = list(scores_dict.values())
    min_v, max_v = min(vals), max(vals)
    return {k: (v - min_v) / (max_v - min_v + 1e-9) for k, v in scores_dict.items()}

# Import dataset if not exists

In [6]:
def save_jsonl(items, path):
    with open(path, "w", encoding="utf-8") as f:
        for d in items:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

def save_qrels(qrels_list, path):
    # qrels_list là list của (query_id, doc_id, rel)
    with open(path, "w", encoding="utf-8") as f:
        # mỗi dòng: query_id \t 0 \t doc_id \t relevance
        for qid, docid, rel in qrels_list:
            f.write(f"{qid}\t0\t{docid}\t{rel}\n")

def convert_ms_marco(split="validation", out_dir=os.path.join(base_path, "data/ms_marco_val")):
    ds = load_dataset("microsoft/ms_marco", 'v2.1', split=split)
    os.makedirs(out_dir, exist_ok=True)

    # Queries
    queries = [{"id": str(rec["query_id"]), "text": rec["query"]} for rec in ds]

    # Corpus & Qrels
    corpus_map = {}
    qrels = []

    for rec in ds:
        qid = str(rec["query_id"])
        passages = rec["passages"]
        texts = passages["passage_text"]
        selected = passages["is_selected"]

        for idx, text in enumerate(texts):
            if not text:
                continue
            doc_id = f"{qid}_{idx}"
            corpus_map[doc_id] = text
            if selected[idx] == 1:
                qrels.append((qid, doc_id, 1))

    corpus = [{"id": did, "text": txt} for did, txt in corpus_map.items()]

    # Save
    save_jsonl(corpus, os.path.join(out_dir, "corpus.jsonl"))
    save_jsonl(queries, os.path.join(out_dir, "queries.jsonl"))
    save_qrels(qrels, os.path.join(out_dir, "qrels.tsv"))

    print("Done convert MS MARCO split:", split)

# if exists the folder, skip
if not os.path.exists(os.path.join(base_path, "data/ms_marco_val/corpus.jsonl")):
    convert_ms_marco(split="validation", out_dir=os.path.join(base_path, "data/ms_marco_val"))

if not os.path.exists(os.path.join(base_path, "data/ms_marco_train/corpus.jsonl")):
    convert_ms_marco(split="train", out_dir=os.path.join(base_path, "data/ms_marco_train"))


# Implementing retrievers

## Milvus

In [7]:
from pymilvus import MilvusClient
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm
import os

class MilvusRetriever:
    def __init__(self, corpus, storage_path, model_name="sentence-transformers/all-MiniLM-L6-v2", device="cuda"):
        """
        Phiên bản MilvusRetriever đã được viết lại hoàn toàn,
        sử dụng MilvusClient đơn giản.
        - storage_path: Đường dẫn đến file database, ví dụ: "./milvus_colab.db"
        """
        # --- 1. Khởi tạo Client ---
        # Dòng này thay thế toàn bộ phần server phức tạp
        print(f"Initializing MilvusClient with data file at: {storage_path}")
        self.client = MilvusClient(storage_path)

        self.model = SentenceTransformer(model_name, device=device)
        self.collection_name = "ms_marco_collection"

        self.client.drop_collection(self.collection_name)

        # --- 2. Indexing dữ liệu (nếu cần) ---
        self._create_and_index_collection(corpus)
        # if not self.client.has_collection(self.collection_name):
        #     print(f"Collection '{self.collection_name}' not found. Creating and indexing...")
        #     self._create_and_index_collection(corpus)
        # else:
        #     print(f"Found existing collection '{self.collection_name}'.")
        #     self.client.create_index(self.collection_name)

    def _create_and_index_collection(self, corpus):
        embedding_dim = self.model.get_sentence_embedding_dimension()

        self.client.create_collection(
            collection_name=self.collection_name,
            dimension=embedding_dim,
            primary_field_name="pk",
            id_type="int",
            auto_id=True,
            vector_field_name="embedding",
            metric_type="IP",
            enable_dynamic_field=True
        )
        print("Collection created with auto-incrementing integer PK.")

        print("Encoding corpus... This may take a very long time.")
        texts = [doc["text"] for doc in corpus]
        embeddings = self.model.encode(
            texts, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True
        )

        # Lặp qua và insert theo từng chunk.

        print("Inserting data into Milvus in chunks to save memory...")
        chunk_size = 50000 # Mỗi lần xử lý 50,000 document

        for i in tqdm(range(0, len(corpus), chunk_size), desc="Inserting chunks"):
            # Lấy ra một chunk nhỏ
            corpus_chunk = corpus[i:i+chunk_size]
            embeddings_chunk = embeddings[i:i+chunk_size]

            # Tạo list data chỉ cho chunk này
            data_chunk_to_insert = [
                {
                    "doc_id": doc["id"],
                    "embedding": embeddings_chunk[j]
                }
                for j, doc in enumerate(corpus_chunk)
            ]

            # Insert ngay chunk này vào Milvus
            self.client.insert(
                collection_name=self.collection_name,
                data=data_chunk_to_insert
            )

        # Sau khi insert xong tất cả các chunk
        print(f"Inserted {len(corpus)} documents.")

        print("Creating index...")
        self.client.create_index(self.collection_name)
        print("✅ Index created successfully.")

    def search_batch(self, queries, k=10, batch_size=64):
        print("Encoding queries for Milvus...")
        q_vecs = self.model.encode(
            queries, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True,
            batch_size=batch_size, device=self.model.device
        )

        print("Searching in Milvus...")
        milvus_results = self.client.search(
            collection_name=self.collection_name,
            data=q_vecs.tolist(), # Chuyển sang list
            limit=k,
            output_fields=["doc_id"] # Chỉ cần lấy doc_id
        )

        # Chuyển đổi kết quả về format mà hàm evaluate mong muốn
        final_results = []
        for hits in milvus_results:
            query_res = []
            for hit in hits:
                # hit['id'] bây giờ chính là doc_id gốc
                # hit['distance'] là score
                # query_res.append((hit['id'], hit['distance']))
                original_doc_id = hit['entity']['doc_id']
                score = hit['distance']
                query_res.append((original_doc_id, score))
            final_results.append(query_res)
        return final_results

## Pyserini

In [8]:
import os
import json
from tqdm import tqdm

# --- Định nghĩa các đường dẫn ---
# File corpus đã được đổi tên key cho Pyserini
pyserini_corpus_path = os.path.join(base_path, "data/ms_marco_val/pyserini_corpus.jsonl")
# Thư mục chứa index của Pyserini
pyserini_index_path = os.path.join(base_path, "data/ms_marco_val/pyserini_index")

# --- Bước 2.1: Chuyển đổi corpus (chỉ chạy nếu file chưa tồn tại) ---
# Pyserini yêu cầu key 'text' phải được đổi tên thành 'contents'
if not os.path.exists(pyserini_corpus_path):
    print("Converting corpus to Pyserini format (renaming 'text' to 'contents')...")
    with open(os.path.join(base_path, "data/ms_marco_val/corpus.jsonl"), 'r') as infile, \
         open(pyserini_corpus_path, 'w') as outfile:
        for line in tqdm(infile, desc="Converting"):
            data = json.loads(line)
            pyserini_data = {'id': data['id'], 'contents': data['text']}
            outfile.write(json.dumps(pyserini_data) + '\n')
    print(f"✅ Converted corpus saved to {pyserini_corpus_path}")
else:
    print(f"Pyserini corpus file already exists at {pyserini_corpus_path}")

# --- Bước 2.2: Tạo Index (chỉ chạy nếu thư mục index chưa tồn tại) ---
# Đây là bước tốn thời gian, tương tự như việc tạo embeddings
if not os.path.exists(pyserini_index_path):
    print("\nCreating Pyserini index... This may take a long time.")
    # Gọi Pyserini từ command line để tạo index
    !python -m pyserini.index.lucene \
      --collection JsonCollection \
      --input {pyserini_corpus_path} \
      --index {pyserini_index_path} \
      --generator DefaultLuceneDocumentGenerator \
      --threads 1 # Giữ 1 thread để tránh lỗi RAM trên Colab
    print(f"✅ Index created at {pyserini_index_path}")
else:
    print(f"\nPyserini index already exists at {pyserini_index_path}")

Pyserini corpus file already exists at /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/pyserini_corpus.jsonl

Creating Pyserini index... This may take a long time.
/usr/bin/python3: No module named pyserini.index.lucene
✅ Index created at /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/pyserini_index


In [9]:
from pyserini.search.lucene import LuceneSearcher
from tqdm import tqdm

class PyseriniRetriever:
    """Retriever sử dụng Pyserini (Lucene/Elasticsearch core)."""
    def __init__(self, index_path):
        print(f"Loading Pyserini index from {index_path}...")
        self.searcher = LuceneSearcher(index_path)
        # Pyserini mặc định dùng BM25
        print("✅ PyseriniRetriever (BM25 Premium) initialized.")

    def search(self, query, k=10):
        """Tìm kiếm cho một query."""
        hits = self.searcher.search(query, k=k)
        return [(hit.docid, hit.score) for hit in hits]

    def search_batch(self, queries, k=10, batch_size=64, threads=4):
        """Tìm kiếm batch hiệu quả."""
        # Pyserini có sẵn hàm batch_search rất mạnh
        # Nó tự động xử lý đa luồng trên CPU
        results_dict = self.searcher.batch_search(
            queries=queries,
            q_ids=[str(i) for i in range(len(queries))], # Cần q_ids giả để map kết quả
            k=k,
            threads=threads
        )

        # Chuyển đổi kết quả về đúng format mà hàm evaluate mong muốn
        final_results = []
        for i in range(len(queries)):
            hits = results_dict.get(str(i), [])
            final_results.append([(hit.docid, hit.score) for hit in hits])

        return final_results

ModuleNotFoundError: No module named 'pyserini.search.lucene'

# Evaluation

In [10]:
corpus = load_jsonl(os.path.join(base_path, "data/ms_marco_val/corpus.jsonl"))
queries = load_jsonl(os.path.join(base_path, "data/ms_marco_val/queries.jsonl"))
qrels = load_qrels(os.path.join(base_path, "data/ms_marco_val/qrels.tsv"))

# take only 1000 queries
queries = queries[:1000]


In [11]:
import math
import json
import numpy as np
from tqdm import tqdm

def recall_at_k(run, qrels, k):
    # run: list of (docid, score)
    relevant = {d for d, rel in qrels.items() if rel > 0}
    retrieved = {d for d, _ in run[:k]}
    if not relevant:
        return 0.0
    return len(relevant & retrieved) / len(relevant)

def dcg_at_k(run, qrels, k):
    import math
    return sum(
        (2**qrels.get(doc, 0) - 1) / math.log2(idx + 2)
        for idx, (doc, _) in enumerate(run[:k])
    )

def ndcg_at_k(run, qrels, k):
    ideal = sorted(qrels.values(), reverse=True)[:k]
    idcg = sum((2**rel - 1) / math.log2(idx + 2) for idx, rel in enumerate(ideal))
    if idcg == 0:
        return 0.0
    return dcg_at_k(run, qrels, k) / idcg

In [12]:
import json
import numpy as np
from tqdm import tqdm
import time

corpus_map = {doc['id']: doc['text'] for doc in corpus}

def evaluate(retriever, queries, qrels, k=10, batch_size=32, save_path=None):
    ndcgs, recalls = [], []
    all_outputs = []

    q_texts = [q["text"] for q in queries]
    q_ids = [q["id"] for q in queries]
    all_runs = []

    start_time = time.time()
    if retriever.__class__.__name__ == 'BM25Retriever':
        for qtext in tqdm(q_texts, desc=f"BM25 Search"):
            all_runs.append(retriever.search(qtext, k=k))
    else:
        all_runs = retriever.search_batch(q_texts, k=k, batch_size=batch_size)

    end_time = time.time()
    total_time = end_time - start_time
    avg_latency_ms = (total_time / len(queries)) * 1000

    for i in tqdm(range(len(queries)), desc="Finalizing results"):
        qid = q_ids[i]
        qtext = q_texts[i]
        run = all_runs[i]
        qrel = qrels.get(qid, {})

        ndcgs.append(ndcg_at_k(run, qrel, k))
        recalls.append(recall_at_k(run, qrel, k))

        all_outputs.append({
            "query_id": qid,
            "query_text": qtext,
            "results": [{"doc_id": docid, "text": corpus_map.get(docid, "CONTENT NOT FOUND"), "score": score} for docid, score in run]
        })

    print("Total time: ", total_time)
    print("Average latency: ", avg_latency_ms)

    if save_path:
        print(f"Saving all {len(all_outputs)} query results to {save_path}...")
        with open(save_path, "w", encoding="utf-8") as f:
            for record in all_outputs:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        print(f"Saved retrieval results successfully.")

    # Trả về kết quả trung bình
    return float(np.mean(ndcgs)), float(np.mean(recalls)), all_outputs

In [ ]:
# --- Định nghĩa đường dẫn để lưu database của Milvus trên Google Drive ---
# Nó sẽ tạo một file duy nhất, không phải một thư mục
milvus_db_path = os.path.join(base_path, "data/milvus_colab.db")

# --- Khởi tạo MilvusRetriever ---
# Lần đầu chạy, nó sẽ encode và index toàn bộ corpus, sẽ rất lâu.
# Những lần sau, nó sẽ chỉ load lại từ file .db trên Drive, sẽ nhanh hơn.
milvus_ret = MilvusRetriever(corpus, storage_path=milvus_db_path)

Initializing MilvusClient with data file at: /content/drive/MyDrive/CapstoneProject/Retriever/data/milvus_colab.db
Collection created with auto-incrementing integer PK.
Encoding corpus... This may take a very long time.


Batches:   0%|          | 0/31531 [00:00<?, ?it/s]

Inserting data into Milvus in chunks to save memory...


Inserting chunks:   0%|          | 0/21 [00:00<?, ?it/s]

In [ ]:
print("\n--- Evaluating MilvusRetriever ---")
milvus_ndcg, milvus_recall, milvus_all_outputs = evaluate(
    milvus_ret,
    queries, # Dùng 1000 queries đã cắt
    qrels,
    k=10,
    save_path=os.path.join(base_path, "data/ms_marco_val/milvus_results.jsonl")
)

print(f"\n✅ Milvus Results: nDCG@10 = {milvus_ndcg:.4f}, Recall@10 = {milvus_recall:.4f}")

In [ ]:
import gc
del milvus_ret
gc.collect()

In [ ]:
# --- Khởi tạo PyseriniRetriever ---
pyserini_index_path = os.path.join(base_path, "data/ms_marco_val/pyserini_index")
pyserini_ret = PyseriniRetriever(pyserini_index_path)


# --- Chạy đánh giá ---
print("\n--- Evaluating Pyserini (BM25 Premium) ---")
# Hàm evaluate của mày sẽ tự động dùng `search_batch` vì nó tồn tại
pyserini_ndcg, pyserini_recall, pyserini_all_outputs = evaluate(
    pyserini_ret,
    queries, # Dùng 1000 queries đã cắt
    qrels,
    corpus_map, # Dùng corpus_map đã tạo
    k=10,
    save_path=os.path.join(base_path, "data/ms_marco_val/pyserini_results.jsonl")
)

print(f"\n✅ Pyserini (BM25 Premium) Results: nDCG@10 = {pyserini_ndcg:.4f}, Recall@10 = {pyserini_recall:.4f}")

In [ ]:
# ndcg10, recall10, bm25_outputs = evaluate(bm25, queries, qrels, k=10, save_path=os.path.join(base_path, "data/ms_marco_val/bm25_results.jsonl"))
# print(f"nDCG@10={ndcg10}, Recall@10={recall10}")

In [ ]:
# del bm25
# gc.collect()

In [ ]:
# embeddings_path = os.path.join(base_path, "data/ms_marco_val/corpus_embeddings_all-MiniLM-L6-v2.npy")
# dense = DenseRetriever(corpus, embeddings_path=embeddings_path)

In [ ]:
# ndcg10, recall10, dense_outputs = evaluate(dense, queries, qrels, k=10, save_path=os.path.join(base_path, "data/ms_marco_val/dense_results.jsonl"))
# print(f"nDCG@10={ndcg10}, Recall@10={recall10}")

In [ ]:
# del dense
# gc.collect()

In [ ]:
# def combine_and_evaluate_hybrid(bm25_outputs, dense_outputs, qrels, k=10, save_path=None, method='weighted_sum', alpha=0.5, rrf_k=60):
#     """
#     Hàm này không search lại. Nó chỉ lấy 2 bộ kết quả đã có,
#     trộn chúng lại bằng phương pháp được chọn ('weighted_sum' hoặc 'rrf'),
#     tính toán metrics và lưu file.
#     """
#     print(f"Combining BM25 and Dense results for Hybrid evaluation using '{method}' method...")
#     ndcgs, recalls = [], []
#     hybrid_outputs = []

#     assert len(bm25_outputs) == len(dense_outputs)

#     for i in tqdm(range(len(bm25_outputs)), desc=f"Combining Hybrid results ({method})"):
#         bm_res = bm25_outputs[i]
#         dn_res = dense_outputs[i]

#         qid = bm_res["query_id"]
#         qtext = bm_res["query_text"]
#         qrel = qrels.get(qid, {})

#         run = []

#         if method == 'weighted_sum':
#             bm_results_dict = {r['doc_id']: r['score'] for r in bm_res['results']}
#             dn_results_dict = {r['doc_id']: r['score'] for r in dn_res['results']}

#             bm_norm = normalize_scores(bm_results_dict)
#             dn_norm = normalize_scores(dn_results_dict)
#             all_ids = set(bm_norm.keys()) | set(dn_norm.keys())
#             scores = {
#                 doc_id: alpha * dn_norm.get(doc_id, 0) + (1 - alpha) * bm_norm.get(doc_id, 0)
#                 for doc_id in all_ids
#             }
#             run = sorted(scores.items(), key=lambda item: item[1], reverse=True)[:k]

#         elif method == 'rrf':
#             bm_ranks = {r['doc_id']: i + 1 for i, r in enumerate(bm_res['results'])}
#             dn_ranks = {r['doc_id']: i + 1 for i, r in enumerate(dn_res['results'])}

#             all_ids = set(bm_ranks.keys()) | set(dn_ranks.keys())

#             rrf_scores = {}
#             for doc_id in all_ids:
#                 score = 0.0
#                 if doc_id in bm_ranks:
#                     score += 1.0 / (rrf_k + bm_ranks[doc_id])
#                 if doc_id in dn_ranks:
#                     score += 1.0 / (rrf_k + dn_ranks[doc_id])
#                 rrf_scores[doc_id] = score

#             run = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)[:k]

#         else:
#             raise ValueError("Method must be 'weighted_sum' or 'rrf'")

#         ndcgs.append(ndcg_at_k(run, qrel, k))
#         recalls.append(recall_at_k(run, qrel, k))

#         hybrid_outputs.append({
#             "query_id": qid,
#             "query_text": qtext,
#             "results": [{"doc_id": docid, "text": corpus_map.get(docid, "CONTENT NOT FOUND"), "score": score} for docid, score in run]
#         })

#     if save_path:
#         print(f"Saving all {len(hybrid_outputs)} hybrid query results to {save_path}...")
#         with open(save_path, "w", encoding="utf-8") as f:
#             for record in hybrid_outputs:
#                 f.write(json.dumps(record, ensure_ascii=False) + "\n")
#         print(f"Saved hybrid results successfully.")

#     return float(np.mean(ndcgs)), float(np.mean(recalls))

In [ ]:
# hybrid_ndcg, hybrid_recall = combine_and_evaluate_hybrid(
#     bm25_outputs,
#     dense_outputs,
#     qrels,
#     method='rrf',
#     rrf_k=60,
#     k=10,
#     save_path=os.path.join(base_path, "data/ms_marco_val/hybrid_rrf_results.jsonl")
# )
# print(f"✅ Hybrid Results: nDCG@10 = {hybrid_ndcg:.4f}, Recall@10 = {hybrid_recall:.4f}")

In [ ]:
# hybrid_ndcg, hybrid_recall = combine_and_evaluate_hybrid(
#     bm25_outputs,
#     dense_outputs,
#     qrels,
#     alpha=0.7,
#     method='weighted_sum',
#     k=10,
#     save_path=os.path.join(base_path, "data/ms_marco_val/hybrid_weighted_results.jsonl")
# )
# print(f"✅ Hybrid Results: nDCG@10 = {hybrid_ndcg:.4f}, Recall@10 = {hybrid_recall:.4f}")

# Đánh giá và giải thích kết quả

BM25 cho ra kết quả rất thấp. Kết quả của BM25 tệ đến mức nó trở thành "nhiễu" cho Hybrid.

BM25 thất bại trong trường hợp này vì 3 lý do chính sau:
## 1. Không hiểu từ đồng nghĩa và ngữ nghĩa (Synonym Problem)

BM25 là một cái máy đếm từ. Nó không có khái niệm về "ý nghĩa".
- Query: "chi phí sinh hoạt ở thành phố Hồ Chí Minh"
- Tài liệu liên quan: "Giá cả cho cuộc sống tại Sài Gòn khá đắt đỏ."
BM25 thấy gì?
Nó sẽ tìm các từ khóa: ["chi", "phí", "sinh", "hoạt", "ở", "thành", "phố", "Hồ", "Chí", "Minh"].
Tài liệu kia không chứa bất kỳ từ khóa nào trong số này (trừ những từ vô nghĩa như "ở").

Đối với BM25, tài liệu này có điểm gần như bằng 0. Nó hoàn toàn không liên quan.

Dense Retriever: biến cả câu query và câu trong tài liệu thành các vector ý nghĩa. Vector của "chi phí sinh hoạt" sẽ rất gần với vector của "giá cả cho cuộc sống". Vector của "thành phố Hồ Chí Minh" sẽ rất gần với vector của "Sài Gòn". Trong không gian vector, hai câu này ở rất gần nhau. Dense Retriever sẽ cho điểm rất cao.
Tóm lại: Với các câu hỏi tự nhiên trong MS MARCO, người dùng diễn đạt ý của họ theo nhiều cách. BM25 chỉ bắt được một cách duy nhất: trùng lặp từ khóa.

## 2. Vấn đề từ vựng không khớp (Vocabulary Mismatch Problem)

Ngay cả khi không có từ đồng nghĩa, người ta vẫn có thể dùng các từ khác nhau để mô tả cùng một thứ.
- Query: "cách sửa lỗi rò rỉ ống nước"
- Tài liệu 1: "hướng dẫn khắc phục sự cố đường ống bị chảy nước"
- Tài liệu 2: "ống nước bị rò rỉ"

BM25 sẽ cho điểm Tài liệu 2 rất cao vì nó chứa chính xác các từ khóa ["ống", "nước", "rò", "rỉ"]. Nó sẽ cho điểm Tài liệu 1 rất thấp, vì các từ ["sửa", "lỗi"] không khớp với ["khắc", "phục", "sự", "cố"]. Dù tài liệu 1 mới là câu trả lời thực sự.

Dense Retriever: nó hiểu rằng "sửa lỗi", "khắc phục sự cố", "hướng dẫn" đều nằm trong một cụm ý nghĩa về "giải quyết vấn đề". Nó sẽ cho điểm cao cho cả hai, và có thể nhận ra Tài liệu 1 có giá trị hơn.

## 3. Sự trừng phạt cho các tài liệu dài (Penalty on Long Documents)

Công thức của BM25 có một thành phần để "trừng phạt" các tài liệu dài hơn, với giả định rằng một từ khóa xuất hiện trong một tài liệu ngắn sẽ có giá trị hơn là trong một tài liệu dài. Điều này thường là đúng, nhưng nó có thể phản tác dụng.

- Query: "lịch sử chiến tranh Việt Nam"
- Tài liệu ngắn: "Chiến tranh Việt Nam là một cuộc chiến dài." (Chứa từ khóa, rất ngắn)
- Tài liệu dài: Một đoạn văn chi tiết dài 500 từ mô tả các giai đoạn của cuộc chiến. (Cũng chứa từ khóa, nhưng rất dài)

BM25 có thể làm gì?
Vì sự trừng phạt độ dài, nó có thể xếp hạng tài liệu ngắn lên trên tài liệu dài, dù tài liệu dài rõ ràng là câu trả lời tốt hơn.

Dense Retriever làm gì?
Nó không quan tâm đến độ dài. Nó chỉ quan tâm đến vector ý nghĩa tổng thể của cả đoạn văn. Nếu vector của đoạn văn dài khớp với ý nghĩa của câu query, nó sẽ được điểm cao.

# Giải pháp
- Cứu Hybrid:
  - Điều chỉnh trọng số alpha: Trong Weighted Sum, thử tăng alpha lên cao hơn (ví dụ 0.8 hoặc 0.9) để cho Dense có trọng số lớn hơn và giảm ảnh hưởng của BM25.
  - Chỉ dùng BM25 như một "bộ lọc": Một kỹ thuật khác là: lấy top 100 từ Dense, sau đó dùng BM25 để xếp hạng lại (re-rank) 100 kết quả đó. Cách này giới hạn phạm vi của BM25 và không để nó làm nhiễu kết quả.

# Kết luận
Đối với bộ dữ liệu MS MARCO, nơi các câu hỏi rất đa dạng và tự nhiên, sự "nông cạn" của BM25 bị bộc lộ hoàn toàn, dẫn đến kết quả rất tệ.